# 📝 Python Documentation, Docstrings & IDE Snippets

Designed by Neil Ghugare 

___

## 📋 Table of Contents

| Section | Title | Description
| :--- | :--- | :--- |
| **1** | **Introduction & Prerequisites** | Scope, expectations, and setup overview.
| **2** | **File Headers & Changelongs** | Creating standardized top-level module documentation and version tracking.
| **3** | **Single-Line & Inline Comments** | Knowing *what* to comment and *what to avoid* (explaining the "why").
| **4** | **Function Docstrings & `help()`** | Writing NumPy-style docstrings and inspecting them interactively.
| **5** | **VS Code Snippets & Automation** | Setting up `hdr` shortcuts to autogenerate headers with `Tab`.
| **6** | **Best Practices Summary** | Quick cheat sheet for maintaining clean scientific code.

___

## 1. Introduction & Prerequisites

Welcome! In scientific computing, code is read for more often than it is written (by your future self, your collaborators, and peer reviewers trying to reproduce your results). 

This notebook covers practical techniques to document your Python scripts, modules, and functions effectively, interact with built-in help engines, and automate your workflow using custom editor snippets.

___

## 2. File Headers & Module Changelogs

Every standalone Python script or library module should start with a top-level **file header docstring**. This provides immediate context for anyone opening the script for the first time—answering who wrote it, why it exists, and what changed recently.

### Standard Header Structure:
- **Metadata:** File name, author, contact email, creation date, and last modified date.
- **Description:** A concise summary of the script's scientific purpose or data pipeline step.
- **Changelog:** A brief list tracking major updates, bug fixes, or algorithm modifications.


In [1]:
"""
===============================================================================
FILE        : fits_spectrum_cleaner.py
AUTHOR      : Jane Doe (jane.doe@university.edu)
PROJECT     : PHYSDepot / Observational Astronomy
CREATED     : 2026-02-15
UPDATED     : 2026-10-24

DESCRIPTION :
    Cleans raw 1D FITS spectroscopic data by removing cosmic ray hits,
    performing continuum normalization, and masking atmospheric absorption lines.

CHANGELOG   :
    - 2026-10-24 (v1.2): Added median filter thresholding for cosmic ray removal.
    - 2026-05-10 (v1.1): Updated continuum fit to use Chebyshev polynomials.
    - 2026-02-15 (v1.0): Initial release.
===============================================================================
"""

print("File header docstring executed as module documentation!")

File header docstring executed as module documentation!


___

## 3. Effective Single-Line & Inline Comments

The cardinal rule of inline comments is: **Explain the WHY, not the WHAT.** 

Bad comments merely rephrase the syntax in plain English. Good comments explain physical units, algorithmic assumptions, non-obvious mathematical choices, or reasons for specific parameter thresholds.


In [ ]:
import numpy as np

# -----------------------------------------------------------------------------
# ❌ BAD COMMENTING: Redundant descriptions of syntax.
# -----------------------------------------------------------------------------
x = 10  # Set x to 10.
flux = 45000.0  # Initialize flux before operating on it.
flux = flux / 1e-17  # Divide flux by 1e-17.


# -----------------------------------------------------------------------------
# ✅ GOOD COMMENTING: Context, physical units, and assumptions.
# ✅ GOOD COMMENTING: Comments are more nicely formatted too (in-line).
# -----------------------------------------------------------------------------
# Convert raw ADU counts to physical flux density [erg s^-1 cm^-2 A^-1].
gain = 1.25             # Detector gain [e-/ADU].
exposure_time = 1200.0  # Total integration time in seconds.
effective_area = 450.0  # Telescope primary mirror area [cm^2].

# Raw data counts from our data collection process (these are fake obviously).
raw_counts = np.array([150, 320, 890, 45000])

# Normalize counts by integration time and area to yield flux.
flux_physical = (raw_counts * gain) / (exposure_time * effective_area)

# Apply 3-sigma clipping threshold to mask saturated detector pixels (> 40,000 ADU).
saturation_mask = raw_counts < 40000
valid_flux = flux_physical[saturation_mask]

print("Valid Flux Values:", valid_flux)

Valid Flux Values: [0.00034722 0.00074074 0.00206019]


___

## 4. Function Docstrings & Python's `help()` Engine

Python functions use **docstrings** (triple-quoted strings `"""..."""` placed directly under the `def` statement) to store embedded documentation.

The scientific community overwhelmingly uses the **NumPy Docstring Standard**, which cleanly organizes:
1. **Description:** Short summary of the function.
2. **Parameters:** Name, type, and physical units of each argument.
3. **Returns:** Type, description, and units of returned outputs.
4. **Notes / References:** Mathematical equations or citations.


In [4]:
def compute_blackbody_lambda(wavelengths, temperature):
    """
    Calculate Planck radiation law spectral radiance per unit wavelength.

    Parameters
    ----------
    wavelengths : float or numpy.ndarray
        Wavelength(s) in meters [m].
    temperature : float
        Blackbody temperature in Kelvin [K].

    Returns
    -------
    numpy.ndarray
        Spectral radiance in units of [W m^-3 sr^-1].

    Notes
    -----
    Uses Planck's law formula:
        I(λ, T) = (2 * h * c^2) / (λ^5 * (exp(h*c / (λ * k_B * T)) - 1))
    """
    h = 6.62607015e-34   # Planck constant [J s].
    c = 2.99792458e8     # Speed of light [m/s].
    k_B = 1.380649e-23   # Boltzmann constant [J/K].

    # Calculate the exponent of the radiance equation: h*c / (wavelength*k_B*T).
    exponent = (h * c) / (wavelengths * k_B * temperature)

    # Then use that to calculate the radiance in units of [W m^-3 sr^-1].
    radiance = (2.0 * h * c**2) / (wavelengths**5 * (np.exp(exponent) - 1.0))
    return radiance

# Inspecting Documentation Interactively
print("--- Calling Native help() ---")
help(compute_blackbody_lambda)


--- Calling Native help() ---
Help on function compute_blackbody_lambda in module __main__:

compute_blackbody_lambda(wavelengths, temperature)
    Calculate Planck radiation law spectral radiance per unit wavelength.

    Parameters
    ----------
    wavelengths : float or numpy.ndarray
        Wavelength(s) in meters [m].
    temperature : float
        Blackbody temperature in Kelvin [K].

    Returns
    -------
    numpy.ndarray
        Spectral radiance in units of [W m^-3 sr^-1].

    Notes
    -----
    Uses Planck's law formula:
        I(λ, T) = (2 * h * c^2) / (λ^5 * (exp(h*c / (λ * k_B * T)) - 1))



There are many ways to do this too. I personally like splitting things up into required parameters and optional parameters. If a function has no required parameters or no optional parameters, I still keep the comment, but underneath I just put "None". 

In [5]:
def compute_blackbody_lambda(wavelengths, temperature):
    """
    Calculate Planck radiation law spectral radiance per unit wavelength.

    Parameters (Required)
    ---------------------
    wavelengths : float or numpy.ndarray
        Wavelength(s) in meters [m].
    temperature : float
        Blackbody temperature in Kelvin [K].

    Parameters (Optional)
    ---------------------
    None. 

    Returns
    -------
    numpy.ndarray
        Spectral radiance in units of [W m^-3 sr^-1].

    Notes
    -----
    Uses Planck's law formula:
        I(λ, T) = (2 * h * c^2) / (λ^5 * (exp(h*c / (λ * k_B * T)) - 1))
    """
    h = 6.62607015e-34   # Planck constant [J s].
    c = 2.99792458e8     # Speed of light [m/s].
    k_B = 1.380649e-23   # Boltzmann constant [J/K].

    # Calculate the exponent of the radiance equation: h*c / (wavelength*k_B*T).
    exponent = (h * c) / (wavelengths * k_B * temperature)

    # Then use that to calculate the radiance in units of [W m^-3 sr^-1].
    radiance = (2.0 * h * c**2) / (wavelengths**5 * (np.exp(exponent) - 1.0))
    return radiance

# Inspecting Documentation Interactively
print("--- Calling Native help() ---")
help(compute_blackbody_lambda)


--- Calling Native help() ---
Help on function compute_blackbody_lambda in module __main__:

compute_blackbody_lambda(wavelengths, temperature)
    Calculate Planck radiation law spectral radiance per unit wavelength.

    Parameters (Required)
    ---------------------
    wavelengths : float or numpy.ndarray
        Wavelength(s) in meters [m].
    temperature : float
        Blackbody temperature in Kelvin [K].

    Parameters (Optional)
    ---------------------
    None.

    Returns
    -------
    numpy.ndarray
        Spectral radiance in units of [W m^-3 sr^-1].

    Notes
    -----
    Uses Planck's law formula:
        I(λ, T) = (2 * h * c^2) / (λ^5 * (exp(h*c / (λ * k_B * T)) - 1))



___

## 5. Efficiency Boost: Custom Editor Snippets (VS Code)

Manually typing a 15-line file header or a 10-line NumPy docstring every time adds friction. You can automate this entirely using **Custom Snippets** in VS Code or Neovim!

### How to set up an `hdr` snippet in VS Code:
1. Open VS Code.
2. Go to **File → Preferences → Configure User Snippets** (or `Cmd+Shift+P` / `Ctrl+Shift+P` → *Snippets: Configure User Snippets*).
3. Select **`python.json`**.
4. Paste the JSON snippet configuration below!

```js
{
    "Header Docstring": {
        "prefix": "hdr",
        "body": [
            "\"\"\"",
            "===============================================================================",
            "FILE        : $TM_FILENAME",
            "AUTHOR      : Your Name (your.email@university.edu)",
            "PROJECT     : PHYSDepot",
            "CREATED     : $CURRENT_YEAR-$CURRENT_MONTH-$CURRENT_DATE",
            "DESCRIPTION : $0",
            "===============================================================================",
            "\"\"\""
        ],
        "description": "Insert standardized file header docstring"
    }
}
```

> 💡 **How it works:**  
> Once saved, whenever you open any `.py` file, type **`hdr`**, and hit **`Tab`**, VS Code will automatically expand the entire header, fill in your current file name and today's date, and place your blinking cursor directly at `DESCRIPTION` (`$0`)!



## 6. Summary Cheat Sheet

| Tool / Concept | Scope | Best Practice |
| :--- | :--- | :--- |
| **File Header** | Top of `.py` files | Tracks author, purpose, date, and a version changelog (`v1.0 -> v1.1`). |
| **Inline Comments** | Inside function bodies | Explains physical units, non-obvious math, and *why* choices were made. |
| **Docstrings** | Functions & Classes | Follow NumPy format (`Parameters`, `Returns`, `Notes`). Include units! |
| **`help()` / `?`** | Interactive shell | Quickly inspect docstrings during analysis without leaving your editor. |
| **IDE Snippets** | Daily coding | Automate headers (`hdr`) and docstring boilerplate to save time. |

---

*Happy documenting! May your future self always understand the code you wrote six months ago!* 🚀✨
